In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
bronze_root = "abfss://bronze@grtadlsdev.dfs.core.windows.net"

latest_folder = sorted([f.name for f in dbutils.fs.ls(bronze_root)])[-1]

bronze_path = f"{bronze_root}/{latest_folder}"

# create a py-dict where keys table names and values are adls paths
files = {
    f.name.replace(".txt", ""): f.path
    for f in dbutils.fs.ls(bronze_path)
    if f.name.endswith(".txt")
}

In [0]:
# create bronze schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# loop over files and write to delta tables
for table_name, path in files.items():
    (
        spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(path)
            .write.format("delta")
            .mode("overwrite")
            .saveAsTable(f"bronze.{table_name}")
    )

In [0]:
# validate the tables
spark.sql("SHOW TABLES IN bronze").show(truncate=False)

# row count
for t in files.keys():
    count = spark.table(f"bronze.{t}").count()
    print(f"{t}: {count} rows")

# peek at the schema
for t in files.keys():
    print(f"\nSchema for bronze.{t}")
    spark.table(f"bronze.{t}").printSchema()